In [ ]:
import pandas as pd
import warnings

# 忽略 pandas 的一些非致命警告，让输出更干净
warnings.filterwarnings('ignore')

# ================= 配置区 =================
RAW_KG_PATH = "kg.csv"  # 确保这个大文件还在你的目录下
SEED_KEYWORD = "Alzheimer"  # ★★★ 修改点：关键词改为阿尔茨海默

def inspect_data():
    print(f"🕵️ 正在侦察 PrimeKG ({RAW_KG_PATH}) 关于 '{SEED_KEYWORD}' 的数据 ...")
    
    try:
        # 读取 PrimeKG (文件很大，low_memory=False 防止 DtypeWarning)
        df = pd.read_csv(RAW_KG_PATH, low_memory=False)
    except FileNotFoundError:
        print(f"❌ 错误：找不到 {RAW_KG_PATH}，请检查路径。")
        return

    # 1. 在原始数据中找 "Alzheimer"
    print(f"   PrimeKG 原始总行数: {len(df)}")
    
    # 模糊匹配任何包含 Alzheimer 的行 (不论大小写)
    mask = (df['x_name'].str.contains(SEED_KEYWORD, case=False, na=False)) | \
           (df['y_name'].str.contains(SEED_KEYWORD, case=False, na=False))
    
    ad_df = df[mask]
    print(f"👉 包含 '{SEED_KEYWORD}' 的相关行数: {len(ad_df)}")
    
    if len(ad_df) == 0:
        print("❌ 奇怪：原始数据里完全找不到 Alzheimer？")
        return

    # 2. 看看这些数据是什么类型的？(基因？表型？药物？)
    print(f"\n📊 {SEED_KEYWORD} 相关数据的 [Type] 分布 (Head <-> Tail):")
    # 创建一个临时列来看连接类型
    ad_df['conn_type'] = ad_df['x_type'] + " <-> " + ad_df['y_type']
    print(ad_df['conn_type'].value_counts().head(10)) # 只看前10种主要类型

    # 3. 看看具体的关系 (Relation)
    print(f"\n🔗 {SEED_KEYWORD} 相关数据的 [Relation] 分布:")
    print(ad_df['relation'].value_counts().head(10))

    # 4. 模拟“去药物化”后的结果
    # ADNI 研究通常关注病理机制（基因、蛋白、表型），而不是药物治疗
    print("\n🔍 模拟过滤药物 (Drug) 后的纯净生物学数据:")
    non_drug_ad = ad_df[
        (ad_df['x_type'] != 'drug') & 
        (ad_df['y_type'] != 'drug') & 
        (ad_df['relation'] != 'indication') & 
        (ad_df['relation'] != 'contraindication')
    ]
    
    print(f"✅ 发现 {len(non_drug_ad)} 条纯生物学关系 (基因/表型/通路)！")
    
    if len(non_drug_ad) > 0:
        print("   👀 预览前 10 条精华数据:")
        print(non_drug_ad[['x_name', 'relation', 'y_name', 'x_type', 'y_type']].head(10))
        
        # 5. 特别检查一下核心基因
        # 在 AD 中，APOE, APP, PSEN1, MAPT (Tau) 是核心，看看有没有
        print("\n🧬 核心 AD 基因检查 (APOE, APP, MAPT):")
        core_genes = ['APOE', 'APP', 'MAPT', 'PSEN1']
        for gene in core_genes:
            gene_hits = non_drug_ad[
                (non_drug_ad['x_name'] == gene) | (non_drug_ad['y_name'] == gene)
            ]
            print(f"   - {gene}: {len(gene_hits)} 条关联")

if __name__ == "__main__":
    inspect_data()

In [ ]:
import pandas as pd
import numpy as np

# ================= 配置区 =================
RAW_KG_PATH = "kg.csv"
OUTPUT_PATH = "primekg_ad_only.csv"  # 输出文件名改为 AD

# ★★★ 核心修改：使用验证成功的“关键词军团” ★★★
# 逻辑：只要包含列表中的任意一个词，就认为是 AD 相关
KEYWORDS = [
    "Alzheimer",
    "Dementia",
    "Memory",
    "Cognitive",
    "Amyloid",
    "Tau",
    "Neurofibrillary"
]

# 必须保留的非药物节点类型 (保持纯净生物学背景)
VALID_TYPES = [
    'disease',
    'gene/protein',
    'effect/phenotype',
    'pathway',
    'biological_process',
    'anatomy'
]


def main():
    print(f"🔹 1. 正在加载原始 PrimeKG ({RAW_KG_PATH}) ...")
    # low_memory=False 防止大文件读取警告
    df = pd.read_csv(RAW_KG_PATH, low_memory=False)

    # ================= 核心筛选逻辑 =================
    print(f"🔹 2. 正在根据关键词军团进行生物学筛选: {KEYWORDS} ...")

    # A. 名字匹配 (升级版)：使用正则表达式进行“或”匹配
    # 'Alzheimer|Dementia|Amyloid...'
    pattern = '|'.join(KEYWORDS)

    mask_name = (df['x_name'].str.contains(pattern, case=False, na=False)) | \
                (df['y_name'].str.contains(pattern, case=False, na=False))

    # B. 类型清洗：剔除 Drug, Indication 等 (只保留 VALID_TYPES)
    mask_type = (df['x_type'].isin(VALID_TYPES)) & (df['y_type'].isin(VALID_TYPES))

    # C. 关系清洗：双重保险，剔除药物关系
    mask_rel = ~df['relation'].isin(['indication', 'contraindication', 'drug_drug', 'off-label use'])

    # 应用筛选
    subset = df[mask_name & mask_type & mask_rel].copy()

    if len(subset) == 0:
        print("❌ 错误：没有找到任何符合条件的数据！请检查关键词。")
        return

    print("\n" + "=" * 40)
    print(f"✅ 初步筛选 AD 相关知识: {len(subset)} 条")
    print("=" * 40)

    # ================= 📊 统计打印区 =================
    print("\n[详细分类统计]")
    rel_counts = subset['relation'].value_counts()
    for rel, count in rel_counts.items():
        note = ""
        if rel == 'disease_protein':
            note = "(疾病-基因)"
        elif rel == 'disease_phenotype_positive':
            note = "(典型症状)"
        elif rel == 'pathway_protein':
            note = "(通路-基因)"
        print(f"  - {rel:<30} : {count:>5} 条 {note}")

    # ================= 补充一步 PPI (基因互作) =================
    # AD 涉及很多蛋白（Amyloid, Tau, APOE），我们需要知道它们之间有没有相互作用
    print("\n🔹 3. 正在补充基因内部互作关系 (PPI) 以完善图结构...")

    # 1. 拿到刚才筛选出的所有基因
    related_genes = set(subset[subset['y_type'] == 'gene/protein']['y_name']) | \
                    set(subset[subset['x_type'] == 'gene/protein']['x_name'])

    print(f"   (识别到 {len(related_genes)} 个 AD 相关关键基因，正在查找它们之间的连线...)")

    if len(related_genes) > 0:
        # 2. 回到原始大表，查找两端都是这些基因，且关系是 protein_protein 的行
        mask_ppi = (
                (df['x_name'].isin(related_genes)) &
                (df['y_name'].isin(related_genes)) &
                (df['relation'] == 'protein_protein')
        )
        ppi_df = df[mask_ppi]
        print(f"   => 成功补充了 {len(ppi_df)} 条基因互作 (PPI) 边")

        # 3. 合并
        final_df = pd.concat([subset, ppi_df]).drop_duplicates()
    else:
        final_df = subset
        print("   => 未发现相关基因，跳过 PPI 补充。")

    # ================= 保存 =================
    print(f"\n🔹 4. 保存最终 AD 子图至: {OUTPUT_PATH}")
    # 同样不使用 sep='\t'，回归标准 CSV
    final_df.to_csv(OUTPUT_PATH, index=False)

    print("-" * 40)
    print(f"🎉 ADNI 专属知识图谱构建完成！总条目: {len(final_df)}")
    print("   包含: AD疾病节点 + 核心症状(Dementia等) + 关键基因(Amyloid/Tau) + 基因互作网络")
    print("-" * 40)


if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import csv
import re
from tqdm import tqdm

# ================= 1. 配置区 =================
CSV_FILES = ['AD.csv', 'mci.csv', 'normal.csv']
PRIMEKG_PATH = "primekg_ad_only.csv"

# 输出文件
OUTPUT_TRIPLETS = "adni_knowledge_triplets.csv"
OUTPUT_E2ID = "adni_kg_entity2id.json"
OUTPUT_R2ID = "adni_kg_relation2id.json"

# ================= 2. 映射与分箱配置 =================

# 2.1 必须排除的列 (防泄露)
EXCLUDE_COLS = {
    'path', 'filename', 'PTID',
    'NC', 'MCI', 'DE', 'COG', 'AD', 'PD', 'FTD', 'VD', 'DLB', 'PDD', 'ADD', 'OTHER',
    'mmse', 'cdr', 'cdrSum'  # 金标准评分，排除
}

# 2.2 PrimeKG 症状映射
SYMPTOM_MAP = {
    "npiq_DEL": "PrimeKG:Delusions",
    "npiq_HALL": "PrimeKG:Hallucinations",
    "npiq_AGIT": "PrimeKG:Agitation",
    "npiq_DEPD": "PrimeKG:Depressivity",
    "npiq_ANX": "PrimeKG:Anxiety",
    "npiq_ELAT": "PrimeKG:Conspicuously happy disposition",
    "npiq_APA": "PrimeKG:Apathy",
    "npiq_DISN": "PrimeKG:Disinhibition",
    "npiq_IRR": "PrimeKG:Irritability",
    "npiq_MOT": "PrimeKG:Restlessness",
}

# 2.3 本地强特征前缀 (必须保留)
IMPORTANT_PREFIXES = [
    "Tesla", "faq_", "his_", "trail", "lm_", "boston",
    "animal", "vege", "digit", "gds", "moca"
]

# 基础属性
base_col_map = {
    "age": "Concept:Age",
    "gender": "Concept:Sex",
    "education": "Concept:Education",
    "race": "Concept:Race"
}


# ================= 3. 数值分箱函数 (防止过拟合的核心) =================
def discretize_val(col_name, val):
    """
    将连续数值离散化，确保相似的数值连到同一个节点。
    """
    try:
        v = float(val)
    except:
        return str(val).strip().lower()

    col_lower = col_name.lower()

    # 1. MRI / 脑容量 (通常是大数值或特定比率)
    if "tesla" in col_lower or "brain" in col_lower:
        # 策略: 如果是大数 (>100)，按 50 分箱；如果是小数，保留1位
        if v > 100:
            return str(int(v // 50 * 50))  # e.g. 1234 -> 1200
        else:
            return f"{v:.1f}"

    # 2. 年龄
    if "age" in col_lower:
        return str(int(v // 5 * 5))  # 每5岁一档

    # 3. 连线测试 (时间秒数)
    if "trail" in col_lower:
        return str(int(v // 10 * 10))  # 每10秒一档

    # 4. 评分量表 (FAQ, GDS, MoCA) -> 直接取整
    if any(x in col_lower for x in ["faq", "gds", "moca", "digit", "boston", "animal"]):
        return str(int(v))

    # 默认保留2位小数
    return f"{v:.2f}"


# ================= 4. 处理逻辑 =================
def load_primekg():
    print(f"🔹 读取 PrimeKG: {PRIMEKG_PATH} ...")
    if not os.path.exists(PRIMEKG_PATH): return []
    triplets = []
    try:
        df = pd.read_csv(PRIMEKG_PATH, low_memory=False)
        cols = df.columns
        x_col = next((c for c in cols if 'x_name' in c or 'head' in c), 'x_name')
        y_col = next((c for c in cols if 'y_name' in c or 'tail' in c), 'y_name')
        r_col = next((c for c in cols if 'relation' in c), 'relation')
        for _, row in tqdm(df.iterrows(), total=len(df), desc="Parsing PrimeKG"):
            triplets.append((f"PrimeKG:{row[x_col]}", str(row[r_col]), f"PrimeKG:{row[y_col]}"))
    except Exception as e:
        print(f"❌ 读取失败: {e}")
    return triplets


def process_adni_files():
    local_triplets = []
    ptid_pattern = re.compile(r"(\d+_S_\d+)")

    for file_name in CSV_FILES:
        if not os.path.exists(file_name): continue
        print(f"    处理 ADNI 文件: {file_name} ...")
        df = pd.read_csv(file_name)

        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Parsing {file_name}"):
            # --- ID 提取 ---
            if 'filename' in row and pd.notna(row['filename']):
                raw = str(row['filename'])
                match = ptid_pattern.search(raw)
                pid = f"Patient:{match.group(1)}" if match else f"Patient:{raw}"
            else:
                continue

            # --- 基础属性 ---
            for col, rel in base_col_map.items():
                if col in row and pd.notna(row[col]):
                    val_bin = discretize_val(col, row[col])
                    local_triplets.append((pid, "has_attribute", f"{rel}:{val_bin}"))

            # --- APOE ---
            if 'apoe' in row and pd.notna(row['apoe']):
                try:
                    aval = int(float(row['apoe']))
                    if aval > 0:
                        gnode = f"Gene:APOE_e4_Copies:{aval}"
                        local_triplets.append((pid, "has_gene_risk", gnode))
                        local_triplets.append((gnode, "risk_factor_for", "PrimeKG:Alzheimer disease"))
                except:
                    pass

            # --- 强特征与 PrimeKG 映射 ---
            for col in df.columns:
                if col in EXCLUDE_COLS or col in base_col_map or col == 'apoe': continue
                val = row[col]
                if pd.isna(val) or str(val) in ['', 'nan', '0', '0.0']: continue

                # A. 症状 (PrimeKG)
                mapped_pkg = None
                for k, v in SYMPTOM_MAP.items():
                    if k in col: mapped_pkg = v; break

                if mapped_pkg:
                    snode = f"Symptom:{col}"
                    # 直连 Shortcut (重要!)
                    local_triplets.append((pid, "exhibits", mapped_pkg))
                    # 细节路径
                    local_triplets.append((pid, "exhibits_detail", snode))
                    local_triplets.append((snode, "severity", f"Level:{int(float(val))}"))
                    continue

                # B. 本地强特征 (MRI, FAQ...)
                is_imp = False
                for prefix in IMPORTANT_PREFIXES:
                    if col.startswith(prefix): is_imp = True; break

                if is_imp:
                    # ★ 使用分箱后的值 ★
                    val_bin = discretize_val(col, val)
                    fnode = f"Feature:{col}_{val_bin}"
                    local_triplets.append((pid, "has_clinical_measure", fnode))

    return local_triplets


def main():
    kt = load_primekg()
    at = process_adni_files()
    all_t = kt + at

    df = pd.DataFrame(all_t, columns=['head', 'relation', 'tail'])
    print(f"原始: {len(df)}")
    df.drop_duplicates(inplace=True)
    print(f"去重后: {len(df)}")

    df.to_csv(OUTPUT_TRIPLETS, index=False)

    ents = sorted(list(set(df['head']) | set(df['tail'])))
    rels = sorted(list(set(df['relation'])))

    with open(OUTPUT_E2ID, 'w') as f: json.dump({e: i for i, e in enumerate(ents)}, f)
    with open(OUTPUT_R2ID, 'w') as f: json.dump({r: i for i, r in enumerate(rels)}, f)
    print("✅ 图谱构建完成！")


if __name__ == "__main__":
    main()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import os
import json
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm import tqdm

# ================= ⚡ 配置区 =================
# 1. 输入文件 (对应 build_adni_primekg.py 的输出)
TRIPLETS_FILE = 'adni_knowledge_triplets.csv'
ENTITY2ID_FILE = 'adni_kg_entity2id.json'
RELATION2ID_FILE = 'adni_kg_relation2id.json'

# 2. 输出文件 (训练好的 Embeddings)
OUTPUT_EMBED = 'adni_kg_embeddings.npy'

# 3. 训练超参数
EMBED_DIM = 128  # 向量维度 (128 是医疗小样本任务的黄金标准)
NUM_EPOCHS = 200  # 训练轮数 (ADNI 数据较干净，200轮通常足够)
BATCH_SIZE = 512  # 批次大小
LR = 0.005  # 学习率
TRAIN_RATIO = 0.9  # 90% 用于训练，10% 用于验证
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"🚀 训练设备: {DEVICE}")


# ================= 🛠️ 数据加载类 =================
class KGDataset(Dataset):
    def __init__(self, triplets_file, entity2id, relation2id):
        print(f"    正在读取图谱文件: {triplets_file} ...")

        # ★★★ 关键点：使用标准 CSV 读取 (逗号分隔) ★★★
        # 这样能完美兼容你刚才 build 脚本生成的格式
        try:
            df = pd.read_csv(triplets_file)
        except Exception as e:
            print(f"❌ 读取 CSV 失败: {e}")
            self.triplets = []
            return

        print(f"    ✅ 成功加载原始数据: {len(df)} 行")

        self.triplets = []
        skipped = 0

        # 将字符串转换为 ID
        # 使用 tqdm 显示进度，让你知道程序没卡死
        for _, row in tqdm(df.iterrows(), total=len(df), desc="Indexing Data"):
            try:
                # 确保读取的是字符串
                h_token = str(row['head']).strip()
                r_token = str(row['relation']).strip()
                t_token = str(row['tail']).strip()

                h = entity2id.get(h_token)
                r = relation2id.get(r_token)
                t = entity2id.get(t_token)

                if h is not None and r is not None and t is not None:
                    self.triplets.append((h, r, t))
                else:
                    skipped += 1
            except Exception:
                skipped += 1
                continue

        self.triplets = torch.LongTensor(self.triplets)

        print(f"    📊 最终有效三元组: {len(self.triplets)}")
        if skipped > 0:
            print(f"    ⚠️ 跳过了 {skipped} 条数据 (可能是 ID 映射不匹配，正常现象)")

        if len(self.triplets) == 0:
            raise ValueError("❌ 错误：没有生成任何有效数据！请检查 entity2id 是否匹配。")

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        return self.triplets[idx]


# ================= 🧠 DistMult 模型 =================
class DistMult(nn.Module):
    def __init__(self, num_entities, num_relations, embed_dim):
        super(DistMult, self).__init__()
        self.num_entities = num_entities

        # 实体嵌入表
        self.ent_emb = nn.Embedding(num_entities, embed_dim)
        # 关系嵌入表
        self.rel_emb = nn.Embedding(num_relations, embed_dim)

        # 初始化参数 (Xavier 初始化有助于快速收敛)
        nn.init.xavier_uniform_(self.ent_emb.weight)
        nn.init.xavier_uniform_(self.rel_emb.weight)

        # 损失函数 (Margin Ranking Loss 是 KGE 的标配)
        self.criterion = nn.MarginRankingLoss(margin=1.0)

    def forward(self, h, r, t):
        # DistMult 核心公式: <h, r, t> = sum(h * r * t)
        h_e = self.ent_emb(h)
        r_e = self.rel_emb(r)
        t_e = self.ent_emb(t)
        score = torch.sum(h_e * r_e * t_e, dim=1)
        return score

    def calculate_loss(self, h, r, t):
        batch_size = h.size(0)

        # 负采样：随机把尾实体换成别的，制造“假”知识
        neg_t = torch.randint(0, self.num_entities, (batch_size,), device=h.device)

        # 计算正样本得分 (应该高)
        pos_score = self.forward(h, r, t)
        # 计算负样本得分 (应该低)
        neg_score = self.forward(h, r, neg_t)

        # 目标：pos_score > neg_score + margin
        target = torch.ones(batch_size, device=h.device)
        loss = self.criterion(pos_score, neg_score, target)
        return loss


# ================= 🏃 主训练循环 =================
def train():
    # 1. 检查文件是否存在
    if not os.path.exists(ENTITY2ID_FILE):
        print(f"❌ 找不到 {ENTITY2ID_FILE}，请先运行 build_adni_primekg.py")
        return

    # 2. 加载 ID 映射
    print("📥 正在加载 ID 映射表...")
    with open(ENTITY2ID_FILE, 'r') as f:
        entity2id = json.load(f)
    with open(RELATION2ID_FILE, 'r') as f:
        relation2id = json.load(f)

    num_ents = len(entity2id)
    num_rels = len(relation2id)
    print(f"    实体总数: {num_ents} | 关系总数: {num_rels}")

    # 3. 准备数据
    dataset = KGDataset(TRIPLETS_FILE, entity2id, relation2id)

    # 划分训练集和验证集
    train_size = int(TRAIN_RATIO * len(dataset))
    test_size = len(dataset) - train_size
    train_data, test_data = random_split(dataset, [train_size, test_size])

    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

    # 4. 初始化模型
    model = DistMult(num_ents, num_rels, EMBED_DIM).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    print(f"\n🔥 开始训练 (共 {NUM_EPOCHS} 轮)...")

    # 5. 循环训练
    for epoch in range(NUM_EPOCHS):
        model.train()
        total_loss = 0

        # 进度条
        progress = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}", leave=False)

        for batch in progress:
            batch = batch.to(DEVICE)
            h, r, t = batch[:, 0], batch[:, 1], batch[:, 2]

            optimizer.zero_grad()
            loss = model.calculate_loss(h, r, t)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            progress.set_postfix({'loss': f"{loss.item():.4f}"})

        avg_loss = total_loss / len(train_loader)

        # 每 1 轮验证一次
        if (epoch) % 1 == 0:
            model.eval()
            test_loss = 0
            with torch.no_grad():
                for batch in test_loader:
                    batch = batch.to(DEVICE)
                    h, r, t = batch[:, 0], batch[:, 1], batch[:, 2]
                    loss = model.calculate_loss(h, r, t)
                    test_loss += loss.item()
            avg_test = test_loss / len(test_loader)
            print(f"Epoch {epoch + 1:03d} | 📉 Train Loss: {avg_loss:.4f} | 🔍 Test Loss: {avg_test:.4f}")

    # 6. 保存结果
    print("\n" + "=" * 40)
    print(f"💾 训练完成！正在保存 Embeddings 至: {OUTPUT_EMBED}")

    # 提取所有实体的向量 (numpy 格式)
    # 形状: [num_entities, 128]
    embeddings = model.ent_emb.weight.detach().cpu().numpy()
    np.save(OUTPUT_EMBED, embeddings)

    print(f"✅ 保存成功！矩阵形状: {embeddings.shape}")
    print("=" * 40)



if __name__ == "__main__":
    train()